# Connectome graph statistics

Interactive companion to `scripts/graph_stats.py`. Run from the repo root (`uv run jupyter lab` after `uv add --dev jupyterlab`, or open in VS Code).

In [ ]:
import os

import numpy as np

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
from wormbrain.connectome import build_connectome, load_config
from wormbrain.controls import CONDITIONS, make_condition
from wormbrain.graphstats import stats

cfg = load_config("configs/connectome.yaml")
real = build_connectome(cfg)
real.n, len(real.classes())

## Real graph against one seed of each control

In [ ]:
import pandas as pd

pd.DataFrame({c: stats(make_condition(real, c, seed=0)) for c in CONDITIONS})

## How much does the glutamate assumption matter?

Glutamate can excite (GLR/NMR receptors) or inhibit (GLC chloride channels). The default treats it as excitatory; see DATA.md.

In [ ]:
for g in ["excitatory", "inhibitory"]:
    s = stats(build_connectome({**cfg, "signs": {**cfg["signs"], "glutamate": g}}))
    print(g, "E:I edges =", round(s["excitatory_edges"] / s["inhibitory_edges"], 1))

## Strongest synapses onto the salt-sensing pathway

In [ ]:
i = real.index(["ASEL", "ASER"])
for k in i:
    out = np.argsort(real.chem[k])[::-1][:8]
    print(
        real.neurons[k],
        "->",
        [(real.neurons[j], int(real.chem[k, j]), int(real.chem_sign[k, j])) for j in out],
    )